# Simulating an acquisition

Every `test_tiffs/` fixture used across `../analyses/` was generated the same way this
notebook demonstrates: `pyS3M.simulation.pattern_source.simulate_acquisition` renders a
synthetic Bayer-mosaic camera stack from a pattern image, with a known injected ground
truth — the same backend `claude/generate_test_fixtures.py` uses internally, exposed here
as a plain, reusable function.

The workflow: load a pattern image → assign a dye to each detected colour → simulate →
write to disk as a real, fittable TIFF stack → round-trip it through the exact same
`AnalysisPipeline.fit` pipeline `01_getting_started_smlm_fitting.ipynb` used, to see
recovered positions against the injected ground truth.

In [ ]:
from pathlib import Path
import tempfile
import json
import numpy as np
import matplotlib.pyplot as plt

from pyS3M.AnalysisPipeline import AnalysisPipeline, FittingConfig
from pyS3M.Constants import AnalysisConfig
from pyS3M.clustering import ClusteringConfig
from pyS3M.simulation import pattern_source
import pyS3M.IOFunctions as IOFunctions
import pyS3M.PlottingBase as PlottingBase

REPO_ROOT = Path.cwd().resolve().parents[1]
PATTERN_PATH = REPO_ROOT / "test_tiffs" / "example_SR_patterns" / "circles_onecolour.png"
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"

cfg = AnalysisConfig(display=False)
pipe = AnalysisPipeline(camera="ximea", config=cfg)
pipe.load_calibration(CALIBRATION_DIR)
plotter = PlottingBase.PublicationPlotter()

fig, ax = plotter.create_figure()
# origin="upper" (not image_plot's microscopy-image default "lower") -- this is a plain
# top-to-bottom PNG, not a physically y-up camera frame; "lower" would flip it.
plotter.image_plot(ax, plt.imread(PATTERN_PATH), origin="upper", title=PATTERN_PATH.name)
plt.show()

## Assign a dye to each detected pattern colour

`pattern_source.mask_from_image` Otsu-thresholds the pattern into one boolean mask per
foreground colour — `circles_onecolour.png` has just one, pure black `(0, 0, 0)`. Candidate
molecule positions are then sampled from within that mask at the requested density.

In [ ]:
from pyS3M.simulation.pattern_source import mask_from_image

masks = mask_from_image(PATTERN_PATH)
{colour: int(mask.sum()) for colour, mask in masks.items()}

### Where does the filter go?

`simulate_acquisition`'s `filter_names` argument is the emission-filter knob — the
parameter users will most want to change for a realistic simulation, so it's worth being
explicit about it here. Pass the same optical filter(s) a real acquisition's emission path
would use (e.g. a bandpass filter in front of the camera), and it is applied to *every*
dye in `colour_to_dye` — a plain dye name, a `Scatterer`, a `NileRedEnvironment`, or a
`SimulatedDye` alike (see `02_simulating_with_custom_dyes.ipynb` for the latter) — before
the per-channel colour ratios (and so the rendered Bayer frames) are computed. Filter names
are looked up in the same spectral database as dye names; list the available ones with
`SpectralFunctions.Spectral_Funcs().filter_names`.

Omitting `filter_names` (the default, `None`) simulates an *unfiltered* emission spectrum
— a fine quick sanity check, but not representative of a real acquisition, where an
emission filter is almost always present and materially shifts the colour split (and so the
recovered average emission wavelength) a dye is detected with. Below we use
`chroma-et670-50m`, a 670/50 nm bandpass real single-molecule setups commonly put in the
red channel for a far-red dye like ATTO 647N — a "reasonable" choice for this simulation.

## Simulate

Calibration maps must be pre-cropped to the field of view the pattern image implies —
`image_fov_camera_pixels` computes that size (the same computation `simulate_acquisition`
does internally, exposed so callers can crop consistently first).

In [ ]:
camera_pixel_size_nm = pipe.pixel_size * 1000.0
width, height = pattern_source.image_fov_camera_pixels(PATTERN_PATH, camera_pixel_size_nm)
print(f"Field of view: {width}x{height} camera px")

bayer_stack, ground_truth, width, height, avg_wl_nm = pattern_source.simulate_acquisition(
    image=PATTERN_PATH,
    colour_to_dye={(0, 0, 0): "ATTO 647N"},
    camera=pipe.camera,
    pixel_size_um=pipe.pixel_size,
    gain_map=pipe.gain_map[:height, :width],
    offset_map=pipe.offset_map[:height, :width],
    variance_map=pipe.variance[:height, :width],
    rqe_map=pipe.rqe[:height, :width],
    n_frames=200,
    density_per_um2=0.1,
    modality="PAINT",
    filter_names=["chroma-et670-50m"],  # emission filter -- see markdown above
    photon_range=(4000.0, 10000.0),  # per-frame photon budget: on every frame, one value is
                                      # drawn uniformly from this range and shared by every
                                      # candidate that's ON that frame (simulate_acquisition's
                                      # own default).
    background_photons=10.0, 
    rng=np.random.default_rng(0),
)
print(f"{bayer_stack.shape[0]} frames simulated, {len(ground_truth)} candidates, "
      f"recommended Peak λ = {avg_wl_nm / 1000.0:.3f} µm")
ground_truth.head()

## A raw simulated frame

In [ ]:
fig, ax = plotter.one_column_plot()
ax = plotter.image_plot(ax, bayer_stack[0], vmin=np.percentile(bayer_stack[0], 0.1),
                       vmax=np.percentile(bayer_stack[0], 99.9))
ax[0].set_title("Frame 0 (raw frame)")
plt.show()

## Write to disk and fit — the same pipeline every analysis notebook uses

Written to a temporary directory here so running this notebook doesn't leave files behind
in the repo; `claude/generate_test_fixtures.py` does the same write (TIFF + `metadata.txt`
ROI sidecar) but into `test_tiffs/` to build a permanent fixture instead.

In [ ]:
out_dir = Path(tempfile.mkdtemp()) / "simulated_acquisition"
out_dir.mkdir(parents=True)
io = IOFunctions.IO_Functions()
io.write_tiff(bayer_stack, out_dir / "sim_MMStack_Default.ome.tif", bit="uint16", pixel_size=pipe.pixel_size)
with open(out_dir / "sim_MMStack_Default_metadata.txt", "w") as f:
    json.dump({"FrameKey-0-0-0": {"ROI": f"0-0-{width}-{height}"}}, f)

fc = FittingConfig(peak_wavelength=avg_wl_nm / 1000.0, pfa=1e-3)
pipe.fit(out_dir, mode="smlm", fitting_config=fc)
locs = pipe.load_localisations(out_dir)
cc = ClusteringConfig(min_cluster_size=2)  # STORM: see 01_getting_started_smlm_fitting.ipynb
sm_db, sf_db = pipe.filter_and_cluster(locs, clustering_config=cc)
print(f"{len(sm_db)} molecules recovered from {len(locs)} raw localisations "
      f"({len(ground_truth)} candidates injected)")

## Recovered positions vs. injected ground truth

Don't expect every injected candidate to reappear here: 200 frames is a short PAINT
acquisition, so a fair number of candidates only bind once (or not at all) in the whole
movie — indistinguishable from noise, and never recoverable regardless of fitting quality.
A real acquisition runs far more frames for exactly this reason.

In [ ]:
pixel_size_nm = pipe.pixel_size * 1000.0
fig, ax = plotter.create_figure()
plotter.scatter_plot(ax, ground_truth["xc_nm"], ground_truth["yc_nm"], xlabel="x (nm)", ylabel="y (nm)",
                      size=30, edgecolor="tab:blue", facecolor="none", label=f"injected ({len(ground_truth)})")
plotter.scatter_plot(ax, sm_db["xc"] * pixel_size_nm, sm_db["yc"] * pixel_size_nm, xlabel="x (nm)", ylabel="y (nm)",
                      size=15, color="tab:red", edgecolor="tab:red", label=f"recovered ({len(sm_db)})")
ax.set_aspect("equal")
ax.set_title("Injected vs. recovered candidate positions")
ax.legend()
plt.show()

## Next: `02_simulating_with_custom_dyes.ipynb`

Every dye above (`"ATTO 647N"`) was a database lookup — next, the same walkthrough for a
dye you define yourself (a custom emission spectrum), including the normalisation lesson
`get_pixel_fractions_rawspectra` teaches and actually wiring the result into
`simulate_acquisition` via `SimulatedDye`.